# Lab 1 - Exercise 2: Pipeline Consolidation

Questo notebook continua direttamente dall'Exercise 1.3. Nel notebook precedente abbiamo verificato che una ResNet-18 con backbone congelato e sola testa finale addestrabile e' una baseline utile, ma non sufficiente a superare la baseline SVM.

Qui l'obiettivo cambia: non vogliamo accumulare nuove celle di training ripetitive, ma consolidare una pipeline riproducibile per lanciare, registrare e confrontare esperimenti.

> **Execution note**
>
> Gli output visibili sono stati prodotti durante le esecuzioni finali o di validazione del laboratorio. Nella versione di consegna i training costosi sono disattivati di default quando sono controllati da flag; checkpoint e artefatti salvati vengono usati per consultazione rapida.

---
---
## Exercise 2: Pipeline Consolidation

Consolidate your implementation. When building applications based on Deep Learning, you will inevitably need to run many experiments. So, it is always a good idea to engineer a reproducible pipeline that allows you to run and re-run experiments with different hyperparameters.

Aspetti richiesti:

- **Model and Loss abstraction**: la pipeline deve poter cambiare modello, loss e optimizer senza riscrivere il training loop.
- **Configuration management**: gli iperparametri non devono essere sparsi nelle celle, ma raccolti in una configurazione.
- **Logging**: le run devono produrre metriche salvabili e confrontabili; WandB e' supportato dalla pipeline; in questa consegna il logging locale e' sufficiente e riproducibile.

## Collegamento con Exercise 1.3

Il notebook precedente prepara una baseline head-only: ResNet-18 pre-addestrata, backbone congelato e nuova testa finale per le 43 classi GTSRB.

Exercise 2 serve a trasformare questa logica in una pipeline riproducibile. Quando cambiano split, batch size, optimizer o loss, dobbiamo poter ricostruire chiaramente cosa e' stato eseguito e confrontare i risultati prodotti dalla pipeline corrente.


In [1]:
from pathlib import Path
import sys

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from dla_lab1.config import experiment_config, load_config
from dla_lab1.experiments import batch_size_for, run_finetuning
from dla_lab1.tracking import (
    experiment_output_dir,
    save_run_artifacts,
    summarize_history,
    wandb_is_available,
)

config = load_config(ROOT / "config" / "config.yaml")

# Lasciato False per evitare di rilanciare training lunghi quando si esegue il notebook.
RUN_TRAINING = False

# WandB e' integrabile nella pipeline; qui manteniamo logging locale per riproducibilita'.
ENABLE_WANDB = False
config["wandb"]["enabled"] = ENABLE_WANDB
config["wandb"]["group"] = "exercise_2_pipeline_consolidation"

print(f"Project root: {ROOT}")
print(f"WandB installed: {wandb_is_available()}")
print(f"WandB enabled for this notebook: {config['wandb']['enabled']}")

Project root: DLA_1
WandB installed: True
WandB enabled for this notebook: True


## Cosa era stato fatto nella versione esplorativa

Nella versione esplorativa l'idea principale era corretta: era stata creata una funzione `train_and_evaluate` e poi erano stati previsti diversi esperimenti cambiando loss e optimizer.

Il limite era la forma: molte configurazioni erano scritte direttamente nelle celle, alcune celle erano quasi duplicate e WandB era gestito manualmente nel notebook. Nel refactoring, questa logica viene spostata in `src/dla_lab1` e controllata da `config/config.yaml`.


In [2]:
planned_head_only_screening = pd.DataFrame([
    ["Adam_CrossEntropy", "CrossEntropy", "Adam", "baseline head-only principale"],
    ["AdamW_CrossEntropy", "CrossEntropy", "AdamW", "alternativa con AdamW"],
    ["SGD_CrossEntropy", "CrossEntropy", "SGD", "optimizer classico con momentum"],
    ["Adam_WeightedCrossEntropy", "WeightedCrossEntropy", "Adam", "class imbalance"],
    ["AdamW_WeightedCrossEntropy", "WeightedCrossEntropy", "AdamW", "class imbalance + AdamW"],
    ["SGD_WeightedCrossEntropy", "WeightedCrossEntropy", "SGD", "class imbalance + SGD"],
    ["Adam_FocalLoss", "FocalLoss", "Adam", "classi difficili"],
    ["AdamW_FocalLoss", "FocalLoss", "AdamW", "classi difficili + AdamW"],
    ["SGD_FocalLoss", "FocalLoss", "SGD", "classi difficili + SGD"],
], columns=["Experiment", "Loss", "Optimizer", "Reason"])

planned_head_only_screening


,Experiment,Loss,Optimizer,Reason
0,Adam_CrossEntropy,CrossEntropy,Adam,baseline head-only principale
1,AdamW_CrossEntropy,CrossEntropy,AdamW,alternativa con AdamW
2,SGD_CrossEntropy,CrossEntropy,SGD,optimizer classico con momentum
3,Adam_WeightedCrossEntropy,WeightedCrossEntropy,Adam,class imbalance
4,AdamW_WeightedCrossEntropy,WeightedCrossEntropy,AdamW,class imbalance + AdamW
5,SGD_WeightedCrossEntropy,WeightedCrossEntropy,SGD,class imbalance + SGD
6,Adam_FocalLoss,FocalLoss,Adam,classi difficili
7,AdamW_FocalLoss,FocalLoss,AdamW,classi difficili + AdamW
8,SGD_FocalLoss,FocalLoss,SGD,classi difficili + SGD


### Commento sullo screening

Questa tabella non riporta risultati: e' solo l'elenco delle prove che la pipeline permette di eseguire in modo ordinato.

La scelta di non duplicare tutte le run nel notebook serve a mantenere il lavoro leggibile. Se una variante viene rilanciata, il risultato deve essere salvato dalla pipeline corrente in `artifacts/runs` oppure tracciato su WandB.


## Nuova architettura della pipeline

La pipeline e' divisa in moduli piccoli. Ogni funzione importante e' commentata nel relativo file `.py`, cosi' il notebook resta leggibile e non diventa un blocco di implementazione.

In [3]:
pipeline_components = pd.DataFrame([
    ["config.py", "load_config, experiment_config", "Legge il YAML e costruisce la configurazione finale della run."],
    ["data.py", "build_dataloaders", "Crea train/validation/test loader e split anti-leakage."],
    ["models.py", "build_classifier", "Crea ResNet-18 e sostituisce la testa finale per 43 classi."],
    ["losses.py", "build_loss", "Seleziona CrossEntropy, WeightedCrossEntropy o FocalLoss."],
    ["train.py", "build_optimizer, train_model", "Crea optimizer/scheduler e gestisce training, validation, checkpoint."],
    ["evaluate.py", "predict, classification_metrics", "Calcola predizioni e metriche di classificazione."],
    ["tracking.py", "save_run_artifacts", "Salva history, config e summary della run in artifacts/runs."],
], columns=["File", "Funzioni principali", "Utilizzo"])

pipeline_components

,File,Funzioni principali,Utilizzo
0,config.py,"load_config, experiment_config",Legge il YAML e costruisce la configurazione f...
1,data.py,build_dataloaders,Crea train/validation/test loader e split anti...
2,models.py,build_classifier,Crea ResNet-18 e sostituisce la testa finale p...
3,losses.py,build_loss,"Seleziona CrossEntropy, WeightedCrossEntropy o..."
4,train.py,"build_optimizer, train_model","Crea optimizer/scheduler e gestisce training, ..."
5,evaluate.py,"predict, classification_metrics",Calcola predizioni e metriche di classificazione.
6,tracking.py,save_run_artifacts,"Salva history, config e summary della run in a..."


## Configuration management

Gli iperparametri principali sono centralizzati in `config/config.yaml`.

Questo rende piu' semplice:

- cambiare batch size;
- cambiare optimizer;
- cambiare loss;
- abilitare o disabilitare WandB;
- rilanciare una run con gli stessi parametri.

In [4]:
available_experiments = pd.DataFrame([
    [name, exp.get("description", ""), exp.get("batch_size_key", "")]
    for name, exp in config["experiments"].items()
], columns=["experiment_name", "description", "batch_size_key"])

available_experiments

,experiment_name,description,batch_size_key
0,feature_svm,ResNet-18 feature extractor plus linear SVM ba...,batch_size_feature_extraction
1,finetune_frozen,Fine-tune only the final classifier head.,batch_size_finetune_frozen
2,ex1_3_head_only_adam_ce,Exercise 1.3 head-only fine-tuning baseline se...,batch_size_head_only_original
3,Adam_CrossEntropy,Preliminary head-only screening run with Adam ...,batch_size_head_only_original
4,AdamW_CrossEntropy,Preliminary head-only screening run with AdamW...,batch_size_head_only_original
5,SGD_CrossEntropy,Preliminary head-only screening run with SGD a...,batch_size_head_only_original
6,Adam_WeightedCrossEntropy,Preliminary head-only screening run with Adam ...,batch_size_head_only_original
7,AdamW_WeightedCrossEntropy,Preliminary head-only screening run with AdamW...,batch_size_head_only_original
8,SGD_WeightedCrossEntropy,Preliminary head-only screening run with SGD a...,batch_size_head_only_original
9,Adam_FocalLoss,Preliminary head-only screening run with Adam ...,batch_size_head_only_original


In [5]:
selected_experiment = "ex1_3_head_only_adam_ce"
exp_cfg = experiment_config(config, selected_experiment)
batch_size = batch_size_for(config, exp_cfg["experiment"]["batch_size_key"])

selected_config_summary = pd.DataFrame([
    ["experiment", selected_experiment],
    ["model", exp_cfg["model"]["name"]],
    ["freeze_backbone", exp_cfg["model"]["freeze_backbone"]],
    ["unfreeze_layer4", exp_cfg["model"].get("unfreeze_layer4", False)],
    ["loss", exp_cfg["training"]["loss"]],
    ["optimizer", exp_cfg["training"]["optimizer"]],
    ["learning_rate", exp_cfg["training"]["learning_rate"]],
    ["batch_size", batch_size],
    ["epochs", exp_cfg["training"]["epochs"]],
    ["scheduler", exp_cfg["training"]["scheduler"]],
], columns=["parameter", "value"])

selected_config_summary

,parameter,value
0,experiment,ex1_3_head_only_adam_ce
1,model,resnet18
2,freeze_backbone,True
3,unfreeze_layer4,False
4,loss,CrossEntropy
5,optimizer,Adam
6,learning_rate,0.001
7,batch_size,128
8,epochs,10
9,scheduler,step


## Logging locale e WandB

Il logging locale e' sempre disponibile e salva i risultati in:

`artifacts/runs/<experiment_name>/`

Dentro questa cartella vengono salvati:

- `history.csv`: loss, accuracy e learning rate per epoca;
- `config_used.yaml`: configurazione usata nella run;
- `summary.json`: riassunto con best epoch e best validation accuracy.

WandB e' supportato dalla pipeline, ma non e' obbligatorio. Per usarlo bisogna installare `wandb`, fare login e impostare `ENABLE_WANDB = True`.

In [6]:
wandb_status = pd.DataFrame([
    ["wandb_installed", wandb_is_available()],
    ["wandb_enabled", config["wandb"]["enabled"]],
    ["wandb_project", config["wandb"]["project"]],
    ["wandb_group", config["wandb"]["group"]],
    ["local_output_dir", experiment_output_dir(config, selected_experiment, root=ROOT)],
], columns=["item", "value"])

wandb_status

,item,value
0,wandb_installed,True
1,wandb_enabled,True
2,wandb_project,DLA_Lab1_Pipeline
3,wandb_group,exercise_2_pipeline_consolidation
4,local_output_dir,DL...


## Setup riproducibile dell'Exercise 1.3

Qui mostriamo quale configurazione viene passata alla pipeline per rilanciare la baseline head-only dell'Exercise 1.3.


In [7]:
exercise_13_setup = pd.DataFrame([
    ["experiment", selected_experiment],
    ["model", exp_cfg["model"]["name"]],
    ["freeze_backbone", exp_cfg["model"].get("freeze_backbone")],
    ["unfreeze_layer4", exp_cfg["model"].get("unfreeze_layer4", False)],
    ["loss", exp_cfg["training"]["loss"]],
    ["optimizer", exp_cfg["training"]["optimizer"]],
    ["learning_rate", exp_cfg["training"]["learning_rate"]],
    ["epochs", exp_cfg["training"]["epochs"]],
    ["batch_size", batch_size],
], columns=["parameter", "value"])

exercise_13_setup


,parameter,value
0,experiment,ex1_3_head_only_adam_ce
1,model,resnet18
2,freeze_backbone,True
3,unfreeze_layer4,False
4,loss,CrossEntropy
5,optimizer,Adam
6,learning_rate,0.001
7,epochs,10
8,batch_size,128


In [8]:
artifact_dir = experiment_output_dir(config, selected_experiment, root=ROOT)
artifact_check = pd.DataFrame([
    ["expected_output_dir", artifact_dir],
    ["history_csv_exists", (artifact_dir / "history.csv").exists()],
    ["summary_json_exists", (artifact_dir / "summary.json").exists()],
    ["config_snapshot_exists", (artifact_dir / "config_used.yaml").exists()],
], columns=["item", "value"])

artifact_check


,item,value
0,expected_output_dir,DL...
1,history_csv_exists,True
2,summary_json_exists,True
3,config_snapshot_exists,True


### Commento sul setup

La pipeline e' pronta per produrre risultati confrontabili. Dopo aver rilanciato un esperimento, la history viene salvata in history.csv e il riepilogo in summary.json dentro la cartella della run.


## Cella di riproducibilita' per rilanciare una run

La cella seguente e' disattivata di default. Per usarla:

1. imposta `RUN_TRAINING = True`;
2. se vuoi WandB, installa `wandb`, fai login e imposta `ENABLE_WANDB = True`;
3. scegli `experiment_to_run`.

La funzione `run_finetuning` salva automaticamente checkpoint e artifact locali. Se WandB e' abilitato, invia anche le metriche online nel progetto/gruppo indicato dal config.

In [9]:
experiment_to_run = "ex1_3_head_only_adam_ce"

if RUN_TRAINING:
    config["wandb"]["enabled"] = ENABLE_WANDB
    config["wandb"]["group"] = "exercise_2_pipeline_consolidation"
    result = run_finetuning(config, experiment_to_run, root=ROOT)
    run_summary = result["artifacts"]["summary"]
    print(f"Run salvata in: {result['artifacts']['output_dir']}")
    pd.DataFrame([run_summary])
else:
    print("Training non eseguito. Imposta RUN_TRAINING = True per rilanciare la run.")

  0%|          | 0/163 [00:13<?, ?it/s]

  0%|          | 0/46 [00:10<?, ?it/s]

Epoch 01 | lr=1.00e-03 | train_loss=1.9356 train_acc=0.4826 | val_loss=2.1150 val_acc=0.4050


  0%|          | 0/163 [00:00<?, ?it/s]

  0%|          | 0/46 [00:00<?, ?it/s]

...

  0%|          | 0/163 [00:00<?, ?it/s]

  0%|          | 0/46 [00:00<?, ?it/s]

Epoch 10 | lr=1.00e-05 | train_loss=0.6148 train_acc=0.8274 | val_loss=2.0463 val_acc=0.4655


epoch,▁▂▃▃▄▅▆▆▇█
learning_rate,████▂▂▂▂▁▁
train/accuracy,▁▅▇▇██████
train/loss,█▃▂▂▁▁▁▁▁▁
validation/accuracy,▁▆▇▆▇█████
validation/loss,█▁▁▆▃▃▄▄▃▃
best_val_acc,0.46667
epoch,10
learning_rate,1e-05
train/accuracy,0.82738
train/loss,0.61484


Run salvata in: DLA_1\artifacts\runs\ex1_3_head_only_adam_ce


## Risultati salvati dalla pipeline nuova

La pipeline non deve solo allenare: deve lasciare traccia di cosa e' stato eseguito. Gli artifact correnti della nuova pipeline sono salvati in `artifacts/runs/<experiment_name>/` e includono `history.csv`, `config_used.yaml` e `summary.json`.

| Run | Ruolo | Best epoch | Best validation accuracy | Ultima train accuracy | Ultima validation accuracy |
|---|---|---:|---:|---:|---:|
| `ex1_3_head_only_adam_ce` | baseline fine-tuning head-only | 6 | 0.4665 | 0.8272 | 0.4653 |
| `ex3_1_layer4_unfrozen` | fine-tuning selettivo di `layer4` | 15 | 0.7584 | 1.0000 | 0.7584 |
| `ex3_1_layer4_conservative_aug` | `layer4` + augmentation conservativa | 9 | 0.7541 | 0.9979 | 0.7512 |
| `ex3_1_layer4_conservative_ls005` | `layer4` + augmentation conservativa + label smoothing | 11 | **0.7741** | 0.9998 | 0.7701 |
| `ex3_1_layer4_aggressive_aug` | `layer4` + augmentation aggressiva | 11 | 0.7433 | 0.9927 | 0.7369 |

Questa tabella non proviene dal notebook storico: riassume gli artifact generati dalla pipeline corrente. Il notebook `Esperimenti_di_prova.ipynb` resta solo un archivio delle run vecchie e non e' usato per decidere il risultato finale.


## Conclusione Exercise 2

La pipeline soddisfa lo scopo dell'esercizio perche' separa configurazione, dati, modello, loss, optimizer, scheduler, logging e valutazione. Questo rende possibile rilanciare esperimenti confrontabili senza copiare celle lunghe o cambiare variabili globali sparse nel notebook.

La proprieta' centrale della pipeline e' che ogni run sia descrivibile con tre oggetti: configurazione usata, curva di training/validation e checkpoint migliore. In questo modo un lettore esterno puo' capire sia **cosa** e' stato provato sia **perche'** un modello e' stato scelto.


## Referenced functions and source files

| Function/class | Defined in | Purpose |
| --- | --- | --- |
| `load_config` | `src/dla_lab1/config.py` | Caricamento configurazione del laboratorio. |
| `build_dataloaders` / `build_retrieval_dataloaders` | `src/dla_lab1/data.py` | Preparazione split, loader e pipeline GTSRB. |
| `run_finetuning` | `src/dla_lab1/experiments.py` | Esecuzione controllata degli esperimenti di fine-tuning. |
| `classification_metrics` | `src/dla_lab1/evaluate.py` | Calcolo metriche di classificazione. |
| `extract_features` | `src/dla_lab1/features.py` | Estrazione feature per baseline e retrieval. |
